# Reproduce Results
This notebook loads the saved models and reports ROC AUC, Precision, and Recall on the train, validation, and test sets.

In [ ]:
import os
import pandas as pd
import sklearn.model_selection
from sklearn.metrics import roc_auc_score, precision_score, recall_score
import joblib
from utils import extract_simple_features, extract_improved_features

## Data Loading and Splitting

In [ ]:
quora_df = pd.read_csv("./quora_data.csv")
A_df, test_df = sklearn.model_selection.train_test_split(quora_df, test_size=0.05, random_state=123)
train_df, val_df = sklearn.model_selection.train_test_split(A_df, test_size=0.05, random_state=123)

## Feature Extraction

In [ ]:
X_train_simp = extract_simple_features(train_df)
X_val_simp = extract_simple_features(val_df)
X_test_simp = extract_simple_features(test_df)

X_train_imp = extract_improved_features(train_df)
X_val_imp = extract_improved_features(val_df)
X_test_imp = extract_improved_features(test_df)

y_train = train_df['is_duplicate'].values
y_val = val_df['is_duplicate'].values
y_test = test_df['is_duplicate'].values

## Evaluation Function

In [ ]:
def evaluate_model(model, X, y):
    preds = model.predict(X)
    probs = model.predict_proba(X)[:, 1]
    return {
        'ROC_AUC': roc_auc_score(y, probs),
        'Precision': precision_score(y, preds),
        'Recall': recall_score(y, preds)
    }

## Load Models & Evaluate

In [ ]:
model_simple = joblib.load('models/model_simple.pkl')
model_improved = joblib.load('models/model_improved.pkl')

results = {
    'Simple_Train': evaluate_model(model_simple, X_train_simp, y_train),
    'Simple_Val': evaluate_model(model_simple, X_val_simp, y_val),
    'Simple_Test': evaluate_model(model_simple, X_test_simp, y_test),
    'Improved_Train': evaluate_model(model_improved, X_train_imp, y_train),
    'Improved_Val': evaluate_model(model_improved, X_val_imp, y_val),
    'Improved_Test': evaluate_model(model_improved, X_test_imp, y_test)
}

results_df = pd.DataFrame(results).T
display(results_df)